# LoCoMo Benchmark

This notebook benchmarks RAG methods on **LoCoMo** (Long-term Conversational Memory),
a QA dataset built from extended conversation sessions.

**Dataset**: [LoCoMo](https://github.com/snap-research/locomo) — loaded from local JSON file.
Questions test memory retrieval over long multi-turn conversations.

**Metrics**:
- **RAGAS** (4 metrics) — retrieval and generation quality
- EM/F1 is **not** used here because conversational answers are typically too long and
  varied for exact-match scoring to be meaningful.

In [ ]:
# --- Environment setup (must run first) ---
from nb_helpers.config import setup_environment, load_config
setup_environment()

from helpers.types import BenchmarkConfig

# --- Configuration ---
METHODS = ["vdb"]                  # Adapter methods to benchmark
MAX_QUESTIONS = 10                  # Number of test questions (0 = all)
TOP_K = 8                           # Chunks to retrieve per query
CHUNK_SIZE = 1200                   # Characters per chunk
CHUNK_OVERLAP = 200                 # Overlap between chunks
LOCOMO_PATH = None                  # Path to locomo JSON (None = default corpus/locomo10.json)
LOCOMO_CATEGORIES = None            # Filter by category, e.g. ["single-hop", "multi-hop"]
SKIP_INDEX = False                  # Set True to reuse existing indexes

config_dict = load_config(overrides={
    "methods": METHODS,
    "max_questions": MAX_QUESTIONS,
    "top_k": TOP_K,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
})
config = BenchmarkConfig.from_dict(config_dict)

BENCHMARK = "locomo"
RUN_CONFIG = {
    "corpus": "locomo",
    "num_questions": MAX_QUESTIONS,
    "top_k": TOP_K,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
}
print(f"Config: {len(METHODS)} methods, {MAX_QUESTIONS} questions, top_k={TOP_K}")

In [ ]:
# --- Load Dataset ---
from nb_helpers.datasets import load_locomo

chunks, testset = load_locomo(
    data_path=LOCOMO_PATH,
    max_questions=MAX_QUESTIONS,
    categories=LOCOMO_CATEGORIES,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
if testset:
    print(f"\nSample question: {testset[0]['question']}")
    print(f"Ground truth: {testset[0]['ground_truth'][:200]}...")
    print(f"Category: {testset[0].get('category', 'N/A')}")

In [ ]:
# --- Run Benchmarks ---
from nb_helpers.pipeline import run_all_methods, save_results

all_results = await run_all_methods(
    methods=METHODS,
    chunks=chunks,
    testset=testset,
    config=config,
    skip_index=SKIP_INDEX,
)
save_results(all_results, BENCHMARK, RUN_CONFIG)

In [ ]:
# --- Evaluate with RAGAS ---
from nb_helpers.metrics import evaluate_all_methods

all_results = await evaluate_all_methods(
    all_results,
    config=config,
    use_ragas=True,
    use_em_f1=False,
)
save_results(all_results, BENCHMARK, RUN_CONFIG)

In [ ]:
# --- Summary Table ---
from nb_helpers.charts import summary_table

summary_table(all_results)

In [ ]:
# --- Charts ---
from nb_helpers.charts import ragas_bar_chart, latency_chart, cost_breakdown_chart

ragas_bar_chart(all_results, title="RAGAS Scores — LoCoMo")
latency_chart(all_results, title="Avg Query Latency — LoCoMo")
cost_breakdown_chart(all_results, title="Cost Breakdown — LoCoMo")